In [1]:
import os
import json
import csv
import random
from collections import Counter

# =====================================================
# CONFIGURAÇÕES
# =====================================================

INPUT_PATH    = "../database/"
M5_INPUT_PATH = "../database/m5/datasets/"

# Três pastas de saída
OUTPUT_PATH_TEST  = "benchmark_prepared_test/"
OUTPUT_PATH_VAL   = "benchmark_prepared_val/"
OUTPUT_PATH_TRAIN = "benchmark_prepared_train/"

# True  = manter apenas séries do comprimento MAIS FREQUENTE
#         (o tamanho que aparece em mais séries) e adicionar
#         "_filtered" ao name do dataset
# False = manter todas as séries
FILTER_MAX_LENGTH = True

# Número alvo de sub-séries ao quebrar CSVs por id
TARGET_NUM_SERIES = 100

# Comprimento mínimo absoluto de uma série no split train (após corte 2*H)
# Séries que ficarem abaixo disso são removidas de TODOS os splits
MIN_SERIES_LENGTH = 32

# Percentual máximo de zeros permitido numa série do M5
# Séries com mais de MAX_ZERO_RATIO zeros são descartadas
M5_MAX_ZERO_RATIO = 0.20

# Para o M5 as séries são muito longas (~1969 pontos). Mantém apenas
# os últimos M5_LAST_N_POINTS de cada série ANTES de separar em
# train/val/test. None = mantém a série completa.
M5_LAST_N_POINTS = 500

for path in [OUTPUT_PATH_TEST, OUTPUT_PATH_VAL, OUTPUT_PATH_TRAIN]:
    os.makedirs(path, exist_ok=True)


# =====================================================
# HELPER: checar se dataset já foi processado
# =====================================================

def dataset_already_exists(dataset_name, horizon):
    """
    Retorna True se o arquivo de test já existe para este dataset/horizon.
    Usa o split test como referência (é o último a ser gerado).
    """
    test_file = os.path.join(
        OUTPUT_PATH_TEST,
        dataset_name,
        f"horizon_{horizon}",
        "dataset.jsonl"
    )
    return os.path.isfile(test_file)


# =====================================================
# DATASETS TSF (formato original)
# =====================================================

DATASETS_TSF = {
    "m3_monthly_dataset.tsf": {
        "name": "m3_monthly",
        "horizon": 18
    },
    "m4_monthly_dataset.tsf": {
        "name": "m4_monthly",
        "horizon": 18
    },
    "nn5_weekly_dataset.tsf": {
        "name": "nn5_weekly",
        "horizon": 8
    },
    "tourism_monthly_dataset.tsf": {
        "name": "tourism_monthly",
        "horizon": 24
    },
    "cif_2016_dataset.tsf": {
        "name": "cif_2016",
        "horizon": 12
    },
    "hospital_dataset.tsf": {
        "name": "hospital",
        "horizon": 12
    },
    "fred_md_dataset.tsf": {
        "name": "fred_md",
        "horizon": 12
    }
}

# =====================================================
# DATASETS CSV (novos)
# =====================================================

DATASETS_CSV = {
    "ETTh.csv": {
        "name": "etth",
        "horizon": 36,
        "target_column": "OT",
        "id_column": "id",
        "date_column": "date",
        "num_series": TARGET_NUM_SERIES
    },
    "weather.csv": {
        "name": "weather",
        "horizon": 36,
        "target_column": "OT",
        "id_column": None,
        "date_column": "date",
        "num_series": TARGET_NUM_SERIES
    },

    # Série única mensal de desemprego dos EUA (FRED/BLS)
    # Uma única série temporal — num_series=1 preserva sem quebrar
    "DesempregoEUA(FRED_BLS).csv": {
        "name": "fred_bls",
        "horizon": 12,
        "target_column": "UNRATE",
        "id_column": None,
        "date_column": "observation_date",
        "num_series": 1
    }
}

# =====================================================
# DATASETS M5
# =====================================================

DATASETS_M5 = {
    "m5": {
        "name": "m5",
        "horizon": 28,
        "train_file": "sales_train_evaluation.csv",
        "test_file":  "sales_test_evaluation.csv",
        "key_cols": ["item_id", "store_id"],
        "meta_cols": ["item_id", "dept_id", "cat_id", "store_id", "state_id"],
    }
}

# =====================================================
# DATASETS WALMART
#
# Formato: cada linha é uma observação semanal.
#   Store, Dept, Date, Weekly_Sales, IsHoliday
#
# Cada série = combinação única de (Store, Dept).
# O dataset tem até 45 lojas × 100 departamentos,
# mas nem toda combinação existe.
#
# Selecionamos NUM_SERIES combinações aleatórias com
# semente fixa (RANDOM_SEED) para garantir que train,
# val e test usem EXATAMENTE as mesmas séries.
#
# Horizon = 5 semanas.
# =====================================================

DATASETS_WALMART = {
    "walmart_sales.csv": {
        "name": "walmart_sales",
        "horizon": 5,
        "target_column": "Weekly_Sales",
        "store_column":  "Store",
        "dept_column":   "Dept",
        "date_column":   "Date",
        "num_series":    200,    # séries aleatórias a selecionar
        "random_seed":   42,     # semente fixa → mesma amostra sempre
    }
}


# =====================================================
# PARSE DO ARQUIVO TSF
# =====================================================

def parse_tsf(file_path):

    sequences = []
    horizons  = []

    with open(file_path, "r", encoding="latin1") as f:

        data_started = False

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith("@data"):
                data_started = True
                continue

            if not data_started:
                continue

            parts = line.split(":")

            if len(parts) == 4:
                _, _, horizon, values = parts
                horizon = int(horizon)

            elif len(parts) == 3:
                _, _, values = parts
                horizon = None

            else:
                continue

            series = []

            for v in values.split(","):

                v = v.strip()

                if v == "?" or v == "":
                    continue

                try:
                    series.append(float(v))
                except:
                    continue

            if len(series) > 0:
                sequences.append(series)
                horizons.append(horizon)

    return sequences, horizons


# =====================================================
# PARSE DO ARQUIVO CSV
# =====================================================

def parse_csv(file_path, target_column, id_column, date_column):

    groups = {}

    with open(file_path, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            if id_column and id_column in row:
                gid = row[id_column].strip()
            else:
                gid = "all"

            raw_value = row[target_column].strip()

            if raw_value == "" or raw_value == "?":
                continue

            try:
                value = float(raw_value)
            except:
                continue

            date_str = row[date_column].strip() if date_column else ""

            if gid not in groups:
                groups[gid] = []

            groups[gid].append((date_str, value))

    for gid in groups:
        groups[gid].sort(key=lambda x: x[0])

    return groups


# =====================================================
# PARSE DO ARQUIVO WALMART
#
# Agrupa as observações por (Store, Dept), ordena por
# Date e retorna um dict:
#   key → lista de floats (Weekly_Sales ordenados por data)
#
# Em seguida amostra aleatoriamente num_series chaves
# com a semente fornecida.
#
# A amostragem acontece ANTES de qualquer outro filtro,
# garantindo que train, val e test usem as mesmas séries.
# =====================================================

def parse_walmart(file_path, target_column, store_column,
                  dept_column, date_column, num_series, random_seed):

    groups = {}  # (store, dept) → [(date_str, value), ...]

    with open(file_path, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            store = row[store_column].strip()
            dept  = row[dept_column].strip()
            key   = f"store{store}_dept{dept}"

            raw_value = row[target_column].strip()

            if raw_value == "" or raw_value == "?":
                continue

            try:
                value = float(raw_value)
            except:
                continue

            date_str = row[date_column].strip() if date_column else ""

            if key not in groups:
                groups[key] = []

            groups[key].append((date_str, value))

    # Ordena cada grupo por data
    for key in groups:
        groups[key].sort(key=lambda x: x[0])

    total_available = len(groups)

    # Amostragem aleatória com semente fixa
    all_keys = sorted(groups.keys())   # sort antes do shuffle → determinismo
    rng = random.Random(random_seed)
    rng.shuffle(all_keys)

    selected_keys = all_keys[:num_series]

    print(
        f"  total_store_dept_combinations={total_available} | "
        f"requested={num_series} | "
        f"selected={len(selected_keys)} | "
        f"random_seed={random_seed}"
    )

    # Extrai apenas os valores (floats) das séries selecionadas
    sequences = []
    for key in selected_keys:
        values = [v for _, v in groups[key]]
        if values:
            sequences.append(values)

    return sequences


# =====================================================
# PARSE DOS ARQUIVOS M5
# =====================================================

def parse_m5(train_path, test_path, key_cols, meta_cols):

    def _make_key(row, key_cols):
        return "_".join(row[c].strip() for c in key_cols)

    def _extract_day_values(row, meta_cols):
        day_cols = sorted(
            [c for c in row.keys()
             if c.startswith("d_") and c[2:].isdigit()],
            key=lambda c: int(c[2:])
        )
        values = []
        for c in day_cols:
            v = row[c].strip()
            if v == "" or v == "?":
                continue
            try:
                values.append(float(v))
            except:
                continue
        return values

    train_map = {}
    with open(train_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key    = _make_key(row, key_cols)
            values = _extract_day_values(row, meta_cols)
            if values:
                train_map[key] = values

    test_map = {}
    with open(test_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key    = _make_key(row, key_cols)
            values = _extract_day_values(row, meta_cols)
            if values:
                test_map[key] = values

    sequences = []
    for key, train_vals in train_map.items():
        test_vals = test_map.get(key, [])
        full = train_vals + test_vals
        if full:
            sequences.append(full)

    return sequences


# =====================================================
# FILTRAR SÉRIES COM EXCESSO DE ZEROS (apenas M5)
# =====================================================

def filter_zero_ratio(sequences, max_zero_ratio=M5_MAX_ZERO_RATIO):

    kept    = []
    removed = 0

    for seq in sequences:
        if len(seq) == 0:
            removed += 1
            continue
        zero_ratio = sum(1 for v in seq if v == 0.0) / len(seq)
        if zero_ratio <= max_zero_ratio:
            kept.append(seq)
        else:
            removed += 1

    return kept, removed


# =====================================================
# QUEBRAR SÉRIE EM N SUB-SÉRIES
# =====================================================

def split_series(values, num_series):

    total = len(values)

    if total == 0:
        return []

    chunk_size = total // num_series

    if chunk_size < 2:
        chunk_size = 2

    actual_num = total // chunk_size

    sub_series = []
    start = 0

    for i in range(actual_num):

        end = start + chunk_size
        sub_series.append(values[start:end])
        start = end

    if start < total:
        sub_series.append(values[start:total])

    return sub_series


# =====================================================
# FILTRAR SERIES PELO COMPRIMENTO MAIS FREQUENTE
# =====================================================

def filter_most_frequent_length(sequences):
    """
    Mantém apenas as séries cujo comprimento é o MAIS FREQUENTE
    no conjunto. Em caso de empate na frequência, escolhe o maior
    comprimento entre os mais frequentes.
    """

    if len(sequences) == 0:
        return sequences, 0

    length_counts = Counter(len(s) for s in sequences)
    max_count     = max(length_counts.values())

    most_frequent_lengths = [
        length for length, count in length_counts.items()
        if count == max_count
    ]

    chosen_length = max(most_frequent_lengths)
    filtered      = [s for s in sequences if len(s) == chosen_length]

    return filtered, chosen_length


# =====================================================
# OBTER COMPRIMENTO MAXIMO
# =====================================================

def get_max_length(sequences):

    if len(sequences) == 0:
        return 0

    return max(len(s) for s in sequences)


# =====================================================
# FILTRAR SÉRIES VIÁVEIS PARA TRAIN
# =====================================================

def filter_viable_sequences(sequences, horizon, min_length=MIN_SERIES_LENGTH):
    """
    Mantém apenas séries que, após o corte mais agressivo
    (train = 2*horizon), ainda tenham pelo menos min_length pontos.
    """

    cut_train = horizon * 2
    viable    = []
    removed   = 0

    for seq in sequences:
        if len(seq) - cut_train >= min_length:
            viable.append(seq)
        else:
            removed += 1

    return viable, removed


# =====================================================
# SALVAR JSONL
# =====================================================

def save_jsonl(sequences, output_file):

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        for seq in sequences:
            f.write(json.dumps({"sequence": seq}) + "\n")


# =====================================================
# SALVAR NAS 3 PASTAS
# =====================================================

def save_all_splits(sequences, dataset_name, horizon):
    """
    Salva o dataset nas 3 pastas.

      train: remove últimos 2*horizon pontos
      val:   remove últimos horizon pontos
      test:  série completa
    """

    splits = [
        ("train", OUTPUT_PATH_TRAIN, 2),
        ("val",   OUTPUT_PATH_VAL,   1),
        ("test",  OUTPUT_PATH_TEST,  0),
    ]

    for split_name, output_base, multiplier in splits:

        cut = horizon * multiplier

        trimmed = list(sequences) if cut == 0 else [seq[:-cut] for seq in sequences]

        if not trimmed:
            print(f"  WARNING: {split_name} - no series")
            continue

        output_file = os.path.join(
            output_base,
            dataset_name,
            f"horizon_{horizon}",
            "dataset.jsonl"
        )

        save_jsonl(trimmed, output_file)

        print(
            f"  {split_name:5s} | "
            f"series={len(trimmed)} | "
            f"min_len={min(len(s) for s in trimmed)} | "
            f"max_len={get_max_length(trimmed)} | "
            f"cut={cut}"
        )


# =====================================================
# PROCESSAMENTO TSF (original)
# =====================================================

print("=" * 60)
print("PROCESSANDO DATASETS TSF")
print("=" * 60)

for file_name, config in DATASETS_TSF.items():

    dataset_name = config["name"]
    horizon      = config["horizon"]

    check_name = dataset_name + "_filtered" if FILTER_MAX_LENGTH else dataset_name

    if dataset_already_exists(check_name, horizon):
        print(f"\nSkipping (already exists): {check_name}/horizon_{horizon}")
        continue

    print(f"\nProcessing: {file_name}")

    dataset_path = os.path.join(INPUT_PATH, file_name)
    sequences, horizons = parse_tsf(dataset_path)

    total_series_original = len(sequences)

    # CIF 2016: filtrar apenas horizon=12
    if dataset_name == "cif_2016":
        sequences = [seq for seq, h in zip(sequences, horizons) if h is None or h == 12]

    total_series_after_horizon_filter = len(sequences)

    if not sequences:
        print("WARNING: No series found after filtering")
        continue

    max_len = get_max_length(sequences)

    if FILTER_MAX_LENGTH:
        sequences, chosen_length = filter_most_frequent_length(sequences)
        print(f"  filter_most_frequent_length | chosen_length={chosen_length} | series_kept={len(sequences)}")
        dataset_name = dataset_name + "_filtered"
        max_len      = get_max_length(sequences)

    sequences, removed = filter_viable_sequences(sequences, horizon)

    if not sequences:
        print("WARNING: No viable series (all too short for train cut)")
        continue

    print(
        f"  original={total_series_original} | "
        f"after_horizon_filter={total_series_after_horizon_filter} | "
        f"viable={len(sequences)} | "
        f"removed_too_short={removed} | "
        f"max_len={max_len} | "
        f"horizon={horizon}"
    )

    save_all_splits(sequences, dataset_name, horizon)


# =====================================================
# PROCESSAMENTO CSV (novos datasets)
# =====================================================

print("\n" + "=" * 60)
print("PROCESSANDO DATASETS CSV")
print("=" * 60)

for file_name, config in DATASETS_CSV.items():

    dataset_name  = config["name"]
    horizon       = config["horizon"]

    check_name = dataset_name + "_filtered" if FILTER_MAX_LENGTH else dataset_name

    if dataset_already_exists(check_name, horizon):
        print(f"\nSkipping (already exists): {check_name}/horizon_{horizon}")
        continue

    print(f"\nProcessing: {file_name}")

    dataset_path  = os.path.join(INPUT_PATH, file_name)
    target_column = config["target_column"]
    id_column     = config["id_column"]
    date_column   = config["date_column"]
    num_series    = config["num_series"]

    groups        = parse_csv(dataset_path, target_column, id_column, date_column)
    all_sequences = []

    for gid, records in groups.items():
        values     = [v for _, v in records]
        chunk_size = len(values) // num_series if len(values) >= num_series else len(values)
        print(
            f"  id={gid} | total_points={len(values)} | "
            f"target_num_series={num_series} | chunk_size={chunk_size}"
        )
        all_sequences.extend(split_series(values, num_series))

    if FILTER_MAX_LENGTH:
        all_sequences, chosen_length = filter_most_frequent_length(all_sequences)
        print(f"  filter_most_frequent_length | chosen_length={chosen_length} | series_kept={len(all_sequences)}")
        dataset_name = dataset_name + "_filtered"

    all_sequences, removed = filter_viable_sequences(all_sequences, horizon)

    if not all_sequences:
        print("WARNING: No viable series generated")
        continue

    print(
        f"  total_ids={len(groups)} | viable_series={len(all_sequences)} | "
        f"removed_too_short={removed} | min_len={min(len(s) for s in all_sequences)} | "
        f"max_len={get_max_length(all_sequences)} | horizon={horizon}"
    )

    save_all_splits(all_sequences, dataset_name, horizon)


# =====================================================
# PROCESSAMENTO M5
# =====================================================

print("\n" + "=" * 60)
print("PROCESSANDO DATASETS M5")
print("=" * 60)

for dataset_key, config in DATASETS_M5.items():

    dataset_name = config["name"]
    horizon      = config["horizon"]

    check_name = dataset_name + "_filtered" if FILTER_MAX_LENGTH else dataset_name

    if dataset_already_exists(check_name, horizon):
        print(f"\nSkipping (already exists): {check_name}/horizon_{horizon}")
        continue

    print(f"\nProcessing: {dataset_key}")

    train_path = os.path.join(M5_INPUT_PATH, config["train_file"])
    test_path  = os.path.join(M5_INPUT_PATH, config["test_file"])
    key_cols   = config["key_cols"]
    meta_cols  = config["meta_cols"]

    sequences = parse_m5(train_path, test_path, key_cols, meta_cols)

    # Mantém apenas os últimos M5_LAST_N_POINTS pontos de cada série
    if M5_LAST_N_POINTS:
        sequences = [seq[-M5_LAST_N_POINTS:] for seq in sequences]
        print(f"  trim_last_points | last_n={M5_LAST_N_POINTS} | series={len(sequences)}")

    total_series_original = len(sequences)

    if total_series_original == 0:
        print("WARNING: No series found in M5 files")
        continue

    # Filtro de zeros (exclusivo M5): remove séries com >20% de zeros
    sequences, removed_zeros = filter_zero_ratio(sequences, M5_MAX_ZERO_RATIO)

    print(
        f"  filter_zero_ratio | "
        f"max_zero_ratio={M5_MAX_ZERO_RATIO:.0%} | "
        f"series_kept={len(sequences)} | "
        f"removed={removed_zeros}"
    )

    if not sequences:
        print("WARNING: No series left after zero-ratio filter")
        continue

    if FILTER_MAX_LENGTH:
        sequences, chosen_length = filter_most_frequent_length(sequences)
        print(f"  filter_most_frequent_length | chosen_length={chosen_length} | series_kept={len(sequences)}")
        dataset_name = dataset_name + "_filtered"

    sequences, removed_short = filter_viable_sequences(sequences, horizon)

    if not sequences:
        print("WARNING: No viable series (all too short for train cut)")
        continue

    print(
        f"  original={total_series_original} | "
        f"after_zero_filter={total_series_original - removed_zeros} | "
        f"viable={len(sequences)} | "
        f"removed_too_short={removed_short} | "
        f"min_len={min(len(s) for s in sequences)} | "
        f"max_len={get_max_length(sequences)} | "
        f"horizon={horizon}"
    )

    save_all_splits(sequences, dataset_name, horizon)


# =====================================================
# PROCESSAMENTO WALMART
#
# Formato: Store, Dept, Date, Weekly_Sales, IsHoliday
# Cada série = combinação única (Store, Dept).
#
# Fluxo:
#   1. parse_walmart: lê, agrupa por (Store, Dept),
#      ordena por Date e sorteia num_series combinações
#      com semente fixa → MESMAS séries em todos os splits
#   2. filter_most_frequent_length (opcional)
#   3. filter_viable_sequences
#   4. save_all_splits → cortes de horizon=5
#        test  = série completa
#        val   = série − 5 semanas
#        train = série − 10 semanas
# =====================================================

print("\n" + "=" * 60)
print("PROCESSANDO DATASETS WALMART")
print("=" * 60)

for file_name, config in DATASETS_WALMART.items():

    dataset_name  = config["name"]
    horizon       = config["horizon"]

    check_name = dataset_name + "_filtered" if FILTER_MAX_LENGTH else dataset_name

    if dataset_already_exists(check_name, horizon):
        print(f"\nSkipping (already exists): {check_name}/horizon_{horizon}")
        continue

    print(f"\nProcessing: {file_name}")

    dataset_path  = os.path.join(INPUT_PATH, file_name)
    target_column = config["target_column"]
    store_column  = config["store_column"]
    dept_column   = config["dept_column"]
    date_column   = config["date_column"]
    num_series    = config["num_series"]
    random_seed   = config["random_seed"]

    # parse_walmart já faz a amostragem aleatória com semente fixa
    sequences = parse_walmart(
        dataset_path,
        target_column,
        store_column,
        dept_column,
        date_column,
        num_series,
        random_seed,
    )

    total_series_original = len(sequences)

    if total_series_original == 0:
        print("WARNING: No series found in Walmart file")
        continue

    if FILTER_MAX_LENGTH:
        sequences, chosen_length = filter_most_frequent_length(sequences)
        print(f"  filter_most_frequent_length | chosen_length={chosen_length} | series_kept={len(sequences)}")
        dataset_name = dataset_name + "_filtered"

    sequences, removed = filter_viable_sequences(sequences, horizon)

    if not sequences:
        print("WARNING: No viable series (all too short for train cut)")
        continue

    print(
        f"  sampled={total_series_original} | "
        f"viable={len(sequences)} | "
        f"removed_too_short={removed} | "
        f"min_len={min(len(s) for s in sequences)} | "
        f"max_len={get_max_length(sequences)} | "
        f"horizon={horizon}"
    )

    save_all_splits(sequences, dataset_name, horizon)


print("\nAll datasets processed successfully.")

PROCESSANDO DATASETS TSF

Skipping (already exists): m3_monthly_filtered/horizon_18

Skipping (already exists): m4_monthly_filtered/horizon_18

Skipping (already exists): nn5_weekly_filtered/horizon_8

Skipping (already exists): tourism_monthly_filtered/horizon_24

Skipping (already exists): cif_2016_filtered/horizon_12

Skipping (already exists): hospital_filtered/horizon_12

Skipping (already exists): fred_md_filtered/horizon_12

PROCESSANDO DATASETS CSV

Skipping (already exists): etth_filtered/horizon_36

Skipping (already exists): weather_filtered/horizon_36

Skipping (already exists): fred_bls_filtered/horizon_12

PROCESSANDO DATASETS M5

Processing: m5
  trim_last_points | last_n=500 | series=30490
  filter_zero_ratio | max_zero_ratio=20% | series_kept=2652 | removed=27838
  filter_most_frequent_length | chosen_length=500 | series_kept=2652
  original=30490 | after_zero_filter=2652 | viable=2652 | removed_too_short=0 | min_len=500 | max_len=500 | horizon=28
  train | series=2652